In [1]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
import pandas as pd
import sklearn.linear_model
import sklearn.metrics
import sklearn.model_selection
import warnings
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

warnings.filterwarnings('ignore')

RANDOM_SEED = 68

def fine_tune_bert(x_train_df, y_train_df, x_val_df, y_val_df, epochs, lr):
    import torch
    model_path = "google-bert/bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)

    for name, param in model.base_model.named_parameters():
        param.requires_grad = False
    for name, param in model.base_model.named_parameters():
        if "pooler" in name:
            param.requires_grad = True

    def tokenize(texts):
        return tokenizer(texts, truncation=True, padding=True, return_tensors="pt")

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    train_labels = torch.tensor([0 if l == "Key Stage 2-3" else 1 for l in y_train_df["Coarse Label"].tolist()])

    model.train()
    batch_size = 16
    train_texts = x_train_df["text"].tolist()
    for epoch in range(epochs):
        print(f"    Training epoch {epoch+1}/{epochs}")
        for i in range(0, len(train_texts), batch_size):
            batch_texts = train_texts[i:i+batch_size]
            batch_labels = train_labels[i:i+batch_size]
            inputs = tokenize(batch_texts)
            inputs["labels"] = batch_labels
            loss = model(**inputs).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    model.eval()
    val_texts = x_val_df["text"].tolist()
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(val_texts), batch_size):
            inputs = tokenize(val_texts[i:i+batch_size])
            probs = softmax(model(**inputs).logits.numpy(), axis=1)[:, 1]
            all_probs.extend(probs)
    val_labels = np.array([0 if l == "Key Stage 2-3" else 1 for l in y_val_df["Coarse Label"].tolist()])
    return sklearn.metrics.roc_auc_score(val_labels, all_probs)


def hyperparameter_selection(x_train_df, y_train_df):
    labels = np.array([0 if l == "Key Stage 2-3" else 1 for l in y_train_df["Coarse Label"].tolist()])
    authors = x_train_df["author"].values
    kf = sklearn.model_selection.GroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    max_auc, best_epochs, best_lr = 0, 0, 0

    for lr in [2e-5, 1e-5]:
        for epochs in [2]:
            print(f"Running LR {lr} Epochs {epochs}...")
            auc_sum = 0
            for train_ind, val_ind in kf.split(x_train_df, labels, authors):
                auc = fine_tune_bert(
                    x_train_df.iloc[train_ind], y_train_df.iloc[train_ind],
                    x_train_df.iloc[val_ind], y_train_df.iloc[val_ind],
                    epochs, lr
                )
                auc_sum += auc
            avg_auc = auc_sum / 5
            print(f"LR {lr} Epochs {epochs} AUC {avg_auc:.6f}")
            if avg_auc > max_auc:
                max_auc, best_epochs, best_lr = avg_auc, epochs, lr

    print("Best AUC:", max_auc)
    print("Best epochs:", best_epochs)
    print("Best LR:", best_lr)
    return best_epochs, best_lr


def test_prediction(x_train_df, y_train_df, x_test_df, best_epochs, best_lr):
    import torch
    model_path = "google-bert/bert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)

    for name, param in model.base_model.named_parameters():
        param.requires_grad = False
    for name, param in model.base_model.named_parameters():
        if "pooler" in name:
            param.requires_grad = True

    def tokenize(texts):
        return tokenizer(texts, truncation=True, padding=True, return_tensors="pt")

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=best_lr)
    train_labels = torch.tensor([0 if l == "Key Stage 2-3" else 1 for l in y_train_df["Coarse Label"].tolist()])
    train_texts = x_train_df["text"].tolist()

    batch_size = 16
    model.train()
    for epoch in range(best_epochs):
        print(f"Training epoch {epoch+1}/{best_epochs}")
        for i in range(0, len(train_texts), batch_size):
            batch_texts = train_texts[i:i+batch_size]
            batch_labels = train_labels[i:i+batch_size]
            inputs = tokenize(batch_texts)
            inputs["labels"] = batch_labels
            loss = model(**inputs).loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    model.eval()
    test_texts = x_test_df["text"].tolist()
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(test_texts), batch_size):
            inputs = tokenize(test_texts[i:i+batch_size])
            probs = softmax(model(**inputs).logits.numpy(), axis=1)[:, 1]
            all_probs.extend(probs)
    np.savetxt('yproba1_test.txt', all_probs)
    print("Predictions saved to yproba1_test.txt")

def main():
    print("start")
    data_dir = '/content/drive/MyDrive/ProjectA'
    x_train_df = pd.read_csv(os.path.join(data_dir, 'x_train.csv'))
    y_train_df = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))
    x_test_df = pd.read_csv(os.path.join(data_dir, 'x_test.csv'))
    #best_epochs, best_lr = hyperparameter_selection(x_train_df, y_train_df)
   
    x_tr, x_val, y_tr, y_val = sklearn.model_selection.train_test_split(
        x_train_df, y_train_df, test_size=0.1, random_state=RANDOM_SEED)
    
    auc = fine_tune_bert(x_tr, y_tr, x_val, y_val, epochs=2, lr=2e-5)
    print("AUC:", auc)
    test_prediction(x_train_df, y_train_df, x_test_df, 2, 2e-5)

if __name__ == "__main__":
    main()

start


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    Training epoch 1/2
    Training epoch 2/2
AUC: 0.8028627450980392


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training epoch 1/2
